# Getting started with digital logic

Welcome to the Introduction to FPGAs workshop!

* _Who is this workshop for?_ all undergraduate students who want to learn how FPGAs work, and how to make FPGAs work for them.
* _What will you learn?_ What the technology is, the motivations for creating and using the technology, common design practices, and usage examples.
* _How will you learn it?_ You will use interactive code examples below, running on your own laptop, with all dependencies pre-installed.
* _Can I learn more?_ Workshop two is in-person: you'll be able to apply the concepts you learn here to digital signals processing. It is on `PLACEHOLDER_DATE` at `PLACEHOLDER_LOCATION`.

## What is an FPGA?

An FPGA (field-programmable gate array) is a chip you can program to do any task you want by wiring up its internal components in different ways. Think of it as a set amount of blocks, each which may be connected to any other, and which may be reconnected again and again so that the same inputs become different outputs on each reconfiguration. These blocks form a **digital circuit**. Therefore, a FPGA is one way of implementing a digital circuit.

Imagine a breadboard where you can rewire every connection instantly by sending it a file.
That's what an FPGA does.

> A digital circuit is a circuit where the inputs and outputs are booleans (only ones and zeroes). When inputs change, we may make the assumption that the outputs will change immediately after (after a very small time). How the outputs change is determined by the "gates" that the inputs pass through. Each gate is represented by a **symbol** in a **circuit diagram**. Certain gates are used more often because they are easily converted from the logical representation to a physical form (electrical circuit); they might be given the labels "basic" or "fundamental".

![An example digital circuit, showcasing the most common visual features of gate symbols using symbols for AND, NOT, NOR, and XOR](./assets/ex-digital-circuit.svg)

The diagram above shows a digital circuit with a single output, named `out` and three inputs `a`, `b`, and `c`. Let's build up our intuition on what each gate does. Well, its symbol tells us:

|Symbol|Name|What it does|
|---|---|---|
|![buffer-symbol](./assets/Buffer_ANSI_Labelled.svg)|Buffer|The output is equal to the input.|
|![and-gate](./assets/AND_ANSI_Labelled.svg)|AND|The output is one only if **all inputs are 1**, else it is zero.|
|![or-gate](./assets/OR_ANSI_Labelled.svg)|OR|The output is one if **any input is 1**, else it is zero.|
|![inverter](./assets/NOT_ANSI_Labelled.svg)|NOT|The output is the opposite of the input. If the input is one, the output is zero; if the input is zero, the output is one.|
|![xor-gate](./assets/XOR_ANSI_Labelled.svg)|XOR (exclusive or)|The output is one only if **exactly one input is 1**.|

![The same example digital circuit, now labelled with numbers: AND is symbol number one, NOT is symbol number two, NOR is symbol number three, and XOR is symbol number four.](./assets/ex-digital-circuit-labelled.svg)

Tracing our path through this digital circuit: some of the gates are listed above, and therefore we know how their outputs are produced. In fact, one way of giving names to each gate's output is to write the gate name and then the inputs of the gate in brackets. But what does gate three do? Gate three is an OR gate with the same circle in front as gate two, a NOT gate. It outputs the opposite of what an OR gate would output for its inputs. It is called a NOR (Not-OR) gate.

> Now, using the same reasoning approach, what does the below gate do? What is its name?
>
> ![nand-gate](./assets/NAND_ANSI_Labelled.svg)

### All digital circuits are tables

How would we figure out what the outputs of this digital circuit are?
One way is with a table of inputs to outputs, called a _truth table_.
Each unique input combination is listed on the left hand side, and the corresponding output is written on the right hand side. How would this look for the output named `out` of the circuit we have been working with?

![The same labelled example circuit.](./assets/ex-digital-circuit-labelled.svg)

Well, if `out = XOR(NOR(AND(a, b), NOT(c)), NOT(c))`, the final truth table is:

|a|b|c|`out`|
|---|---|---|---|
|0|0|0|**1**|
|0|0|1|**1**|
|0|1|0|**1**|
|0|1|1|**1**|
|1|0|0|**1**|
|1|0|1|**1**|
|1|1|0|**1**|
|1|1|1|**0**|

Any digital circuit can be fully described by a truth table.

> **Exercise:** How would you create this truth table? How can you check that it is correct?

### Tables may verify digital circuits

_Or: the interface versus the implementation_

What if we already know what every output should be for a digital circuit, for every possible combination of input values? An engineer must implement this specification.

For example, what is another circuit which could generate the truth table we saw earlier? Could we use fewer gates? (less cost).

|a|b|c|`out`|
|---|---|---|---|
|0|0|0|**1**|
|0|0|1|**1**|
|0|1|0|**1**|
|0|1|1|**1**|
|1|0|0|**1**|
|1|0|1|**1**|
|1|1|0|**1**|
|1|1|1|**0**|

To start answering these questions, we can first prepare a **test bench**. A test bench is a program which verifies that a digital circuit is correct, given a specification, by providing to it different inputs and checking that the output matches the truth table of what we want. The digital circuit is called a _design under test_ (DUT), and the test bench provides different input combinations to it to verify its correctness. A useful digital circuit will have thousands of inputs, so a test bench usually does not check every possible input combination. Some techniques to reduce the number of needed input combinations include:

1. Checking that common input combinations named **test vectors** (so named because the set of zeroes and ones that is one input combination can be represented as a single vector) each result in the correct output.
2. Checking "edge cases", particular input combinations which are selected from engineers' gut feeling.
3. Checking random input combinations. The test bench will calculate the expected outputs on the fly and compare them to the digital circuit's output. Since the test bench is written in more expressive language, it is often more easy to specify this calculation than it is in the digital circuit's HDL description!

> A related verification technique is named _formal verification_. In this method, logical assertions on the output of a digital circuit are given along with the description of the circuit (in the same code). The verification tool then tries to find an input vector which is a counterexample to the assertion.

In the next section we will represent the digital circuit we saw earlier with a structured _hardware description language_ (HDL) named Verilog.

## How does an FPGA work?

An FPGA can be thought of physically as a grid of blocks (which are the gates of the digital circuit) connected by wires. These wires are then connected to each other by a complex network of programmable switches. The switches are programmed by a _bitstream_.

![A diagram of the FPGA architecture: blocks of lookup tables connected by a layer of _routing fabric_](./assets/fpga-architecture.png)

The FPGA is programmed like so: a hardware description language is converted into a netlist (a tree of connected logic gates). The netlist then passes through a place and route (PnR) algorithm, a specialized program provided by the FPGA vendor which understands the location of each block and each programmable switch. The PnR algorithm encodes the state of each switch as either ON or OFF and also encodes the state of each block (what type of digital logic gate it will become) in the same _stream of bits_. This is the bitstream, which is then transmitted to the FPGA chip. The transmission step is done using a serial protocol such as USB (universal serial bus) which is then interpreted by the circuit around the main FPGA chip to program the chip itself; the whole process is specified by the manufacturer.

## Coding using HDL

Let's try to implement the digital circuit, and turn its diagram into concrete Verilog HDL. Along the way, we'll learn Verilog syntax.

![The example circuit: out = XOR(NOR(AND(a, b), NOT(c)), NOT(c))](./assets/ex-digital-circuit-labelled.svg)

Every digital circuit can be thought of as a "black box" exposing only inputs and outputs. Verilog uses the concept of a `module` to represent this black box. A `module` has fixed (static) inputs and outputs which cannot change: after all, we cannot expect to change our circuit on the fly!

```verilog
module digital_example(
    input a,
    input b,
    input c,
    output out
);
    logic gate_1, gate_2, gate_3;
  
    assign gate_1 = a & b;
    assign gate_2 = ~c;
    assign gate_3 = gate_1 ~| gate_2;
    assign out = gate_3 ^ gate_2;

endmodule
```

This code has a lot of new syntax to express what we did!
Let's walk through it.

* First, we declared our inputs and outputs.
* Then, we see the line `logic gate_1, gate_2, gate_3;`. This line declares **signals** — internal wires that carry values between gates. They are not inputs or outputs of the module; they only exist inside the circuit to connect one gate's output to another gate's input.

> **Signals** are the Verilog equivalent of physical wires. You declare them with the `logic` type inside a module (but outside any `always` block). Each signal has a name and carries a single value that can be updated by `assign` statements or inside `always` blocks.

* Next, we `assign`ed some operation on `a`, `b`, `c`, and the signals to each signal. Within these lines are the actual gates: `&` (AND), `~` (NOT), `~|` (NOR), and `^` (XOR). Each `assign` statement connects the output of a gate operation to the named signal, just like soldering a wire from one gate's output pin to the next gate's input pin.

So what we have done is traced the entire path from input to output in text instead of diagram form.

> **Exercise:** how would you implement the following digital circuit in the Verilog?
>
> ![Three inputs named a, b, and c are connected to two outputs named out and carry.](./assets/combinational-exercise-1.svg)
>
> * Tip 1: The OR gate is `|` in Verilog.
> * Tip 2: Connection points (where two wires are connected) are marked as black, filled-in circles.
>
> What did you implement? (How is the output related to the input?)

## Test Benches

Test benches let you verify your circuits in software before programming a real FPGA. They wrap your module in a simulation that feeds it every input combination and checks the outputs against the expected truth table.

Since test benches use simulation-specific Verilog features, it is presented in an appendix. If you would like to learn how to write and run a test bench for the `digital_example` module above, see [Appendix: Test Benches](test_benches.ipynb).


<div style="display:flex; justify-content:space-between;">
  <span></span>
  <span><b>Next: </b> <a href="quartus_install.ipynb">Quartus Install</a></span>
</div>